## <b> Problem Statement:</b>
This dataset includes details of applicants who have applied for loan. The dataset includes details like credit history, loan amount, their income, dependents etc. 

#### <b>Independent Variables:</b>

- Loan_ID

- Gender

- Married

- Dependents

- Education

- Self_Employed

- ApplicantIncome

- CoapplicantIncome

- Loan_Amount

- Loan_Amount_Term

- Credit History

- Property_Area

Dependent Variable (Target Variable):

- Loan_Status

<b>You have to build a model that can predict whether the loan of the applicant will be approved or not on the basis of the details provided in the dataset.</b> 

- <b> Importing require library for performing EDA, Data Wrangling and data cleaning</b>

In [ ]:
from loan_utils import (
    load_data, inspect_data, missing_value_summary, impute_missing,
    encode_categoricals, scale_features, remove_skewness,
    remove_outliers_zscore, split_features_target,
    plot_countplots, plot_boxplots, plot_distributions, plot_box_and_dist,
    train_and_evaluate, find_best_random_state, grid_search_evaluate,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
df = load_data('loan_prediction.xls')

In [191]:
# print('No of Rows:',df.shape[0])
# print('No. of Columns:',df.shape[1])
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [193]:
df.tail()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y
613,LP002990,Female,No,0,Graduate,Yes,4583,0.0,133.0,360.0,0.0,Semiurban,N


In [195]:
df.columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')

# Statistical Analysis

 <b>  Before Going for Statistical exploration of data, first check integrity of data & Missing value </b>

### Data Integrity Check

In [200]:
df.duplicated().sum() # This check any if any duplicated entry exit in dataset 

0

<b> Since dataset is large,  Let check for any entry which is repeated or duplicated in dataset at same date. </b>

#### Comment:
Dataset doesnot contain Any duplicate entry. So Yes To Go !!!

### Datatype Check

In [ ]:
inspect_data(df)

- <b> Comment </b>:
    - In loan application status dataset we have 614 rows with 13 columns including target variable.
    - A Target Variable is 'Loan_Status' having object datatype and It is categorical variable.
    - Gender, Married, Education,Self Employed, Credit History, Loan Status are categorical features.
    - There are three types of datatype dtypes: float64(4), int64(1), object(8)

### Missing value check 

In [ ]:
missing_value_summary(df)

- <b> Comment :</b>
    - 7 out 13 columns contains missing value.
    - As small amount of data is missing so we use mean amd mode to replace with NaN values.
    
<b> Lets explore categorical features before missing value imputation.</b>

### Start with Enlisting Value counts & Sub-categories of different categorial features available

In [213]:
# category=['Gender','Married','Dependents','Education','Self_Employed',
#           'Loan_Amount_Term','Property_Area','Credit_History','Loan_Status']

df.drop("Loan_ID",axis=1,inplace=True)
category=df.select_dtypes(include="object")

for i in category:
    print(i)
    print(df[i].value_counts())
    print('='*100)

Gender
Gender
Male      489
Female    112
Name: count, dtype: int64
Married
Married
Yes    398
No     213
Name: count, dtype: int64
Dependents
Dependents
0     345
1     102
2     101
3+     51
Name: count, dtype: int64
Education
Education
Graduate        480
Not Graduate    134
Name: count, dtype: int64
Self_Employed
Self_Employed
No     500
Yes     82
Name: count, dtype: int64
Property_Area
Property_Area
Semiurban    233
Urban        202
Rural        179
Name: count, dtype: int64
Loan_Status
Loan_Status
Y    422
N    192
Name: count, dtype: int64


In [ ]:
category = df.select_dtypes(include='object')
plot_countplots(df, category.columns, ncols=3, figsize=(20, 20))

- <b> Comment:</b>
     - Out of Total loan application 80 % applicants are Male. <b>We can Explore loan amount for each gender applied and evaluate whether on the same basis loan is approved for each gender or not?</b>
     - Only 20% applicants are self employed. <b> So it will interesting to gain insight on relation between Applicant income and loan approval for non self employed category. We will look to find any benchmark range of Income for loan approval.Another benchmark we will try to find is about loan requirement for these two categories.</b>
     - Nearly 70% are married and 75% of loan applicants are graduates
     - Almost 60% of the applicants have no dependents.
     - Most of applicants come from Semi Urban areas, followed by Urban and Rural areas.
     - 80% people previously have credit history. Normally people having credit history are seen more prone to get loan approval.
     - Nearly 70 % applicant gets loan approved.
    
<b> We can impute categoical variable with mode in that category. For numerical variable we have option of mean and median. If Outliers are to strech then we will impute with median.</b>
    
### Let check outliers for missing values Numerical variable having missing values by plotting boxplot.

In [ ]:
plot_box_and_dist(df, 'LoanAmount')

In [218]:
print("Mean of Loan Amount:",df['LoanAmount'].mean())
print("Median of Loan Amount:",df['LoanAmount'].median())

Mean of Loan Amount: 146.41216216216216
Median of Loan Amount: 128.0


#### Comment -
- The mean is greater than median loan amount.
- Clearly we can see outliers in boxplot and feature is strecth to far in distribution plot.

<b> As extreme outliers are present in feature and for that reason as data is more sensitive to mean we are going to impute missing values in <u>loan amount  with median.</u> </b>

### Imputation of Missing values

#### Imputation details :
1. Missing values in Loan amount is impute with median value.
2. Maximum Loan term is 360 Months so Missing value in Loan amount term is replace with 360 Months.
3. Credit History, Self Employed, dependents, Gender and Married are replace with mode of repective features.

In [ ]:
impute_missing(
    df,
    mode_cols=['Credit_History', 'Self_Employed', 'Dependents', 'Married', 'Loan_Amount_Term'],
    median_cols=['LoanAmount'],
    fill_values={'Gender': 'unknown'},
)

### Missing Value Check After Imputation

In [ ]:
missing_value_summary(df)

#### Comment :
<b> Finally, No Missing Value is Present.

We are Now Yes To Go Further !!!</b>

### Statistical Matrix

In [232]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,614.000000,614.000000,614.000000
mean,5403.459283,1621.245798,145.752443,342.410423,0.855049
std,6109.041673,2926.248369,84.107233,64.428629,0.352339
min,150.000000,0.000000,9.000000,12.000000,0.000000
25%,2877.500000,0.000000,100.250000,360.000000,1.000000
50%,3812.500000,1188.500000,128.000000,360.000000,1.000000
75%,5795.000000,2297.250000,164.750000,360.000000,1.000000
max,81000.000000,41667.000000,700.000000,480.000000,1.000000


#### Comment:
- In Applicant Income & Coapplicant Income Std deviation value is greater than median. So data is spread and skewed.
- Taking 75% and Max rows into consideration we can surely say that Outliers exist in Applicant Income, Coapplicant Income,Loan Amount.
- Since Credit History is Categorical variable there is no significance in different statstical parameter of it.
- Minimum Tenure for Loan is 12 Months and Maximum Loan tenure is 480 Months.
- Minimum Applicant income is 150 and maximum is 81000.


<b> Let dive into exploration of Target and independent feature.</b>

### Target Variable

# Encoding categorical data

In [238]:
df["Dependents"]=df["Dependents"].replace("3+",3)
df["Dependents"]=df["Dependents"].astype(int)

In [ ]:
le = encode_categoricals(df)
df.head()

In [ ]:
X, _ = split_features_target(df)

In [ ]:
import pandas as pd
X_sc, sc = scale_features(X)
X_scale = pd.DataFrame(X_sc, columns=X.columns)
X_scale

In [246]:
np.round(X_scale.describe(),2)

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
count,614.00,614.00,614.00,614.00,614.00,614.00,614.00,614.00,614.00,614.00,614.00
mean,-0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.99,-1.37,-0.74,-0.53,-0.39,-0.86,-0.55,-1.63,-5.13,-2.43,-1.32
25%,0.38,-1.37,-0.74,-0.53,-0.39,-0.41,-0.55,-0.54,0.27,0.41,-1.32
50%,0.38,0.73,-0.74,-0.53,-0.39,-0.26,-0.15,-0.21,0.27,0.41,-0.05
75%,0.38,0.73,0.25,-0.53,-0.39,0.06,0.23,0.23,0.27,0.41,1.22
max,2.76,0.73,2.24,1.89,2.55,12.38,13.70,6.60,2.14,0.41,1.22


# Feature selection and Engineering

## 1. Outliers Detection and Removal

In [253]:
Numerical =['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']

In [ ]:
plot_boxplots(df, Numerical)

<b> From Boxplot we can see outliers exist dataset.</b>

In [ ]:
df = remove_outliers_zscore(df, threshold=3)

##### Data Loss

## 2.  Skewness of features

In [ ]:
plot_distributions(df, Numerical)

In [263]:
df.skew()  #-0.5 --- 0.5--- normal range

Gender              -0.991250
Married             -0.630211
Dependents           1.052106
Education            1.306588
Self_Employed        2.252848
ApplicantIncome      2.148522
CoapplicantIncome    1.350517
LoanAmount           1.151525
Loan_Amount_Term    -2.098806
Credit_History      -1.976043
Property_Area       -0.055332
Loan_Status         -0.822635
dtype: float64

#### Comment :
- <b> Out of all above feature 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount' are skewed which are numerical feature.</b>
- Other features are categorical in nature so skewness is nothing to do with these remaining feature.<u>We will ignore them.</u>
- We will yeo-johnson transformation method.

In [ ]:
skew_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
pt = remove_skewness(df, skew_cols)
df[skew_cols].head()

####  Checking skewness after using yeo-johnson ethod

In [272]:
df.skew()

Gender              -0.991250
Married             -0.630211
Dependents           1.052106
Education            1.306588
Self_Employed        2.252848
ApplicantIncome      0.027981
CoapplicantIncome   -0.191876
LoanAmount           0.048425
Loan_Amount_Term    -2.098806
Credit_History      -1.976043
Property_Area       -0.055332
Loan_Status         -0.822635
dtype: float64

<b> For Numerical variable skewness is within permissible limit.

So Yes To Go Forward !!!
</b>

## 3. Corrleation 

In [276]:
df.corr()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
Gender,1.000000,0.357644,0.174277,0.027503,0.026601,0.051968,0.224699,0.172147,-0.100887,-0.003570,-0.031359,0.001946
Married,0.357644,1.000000,0.329900,0.024817,-0.015779,-0.024783,0.335820,0.181878,-0.127348,0.019308,0.010595,0.089026
Dependents,0.174277,0.329900,1.000000,0.069814,0.044543,0.105994,0.004109,0.131772,-0.087389,-0.020288,0.002327,0.017872
Education,0.027503,0.024817,0.069814,1.000000,-0.007139,-0.176074,0.049739,-0.128715,-0.090523,-0.075217,-0.068596,-0.092658
Self_Employed,0.026601,-0.015779,0.044543,-0.007139,1.000000,0.212260,-0.087338,0.117218,-0.032914,-0.016390,-0.028253,-0.026525
ApplicantIncome,0.051968,-0.024783,0.105994,-0.176074,0.212260,1.000000,-0.360946,0.432154,-0.069429,0.028825,-0.011364,-0.002484
CoapplicantIncome,0.224699,0.335820,0.004109,0.049739,-0.087338,-0.360946,1.000000,0.200081,0.000951,0.006564,-0.074476,0.079344
LoanAmount,0.172147,0.181878,0.131772,-0.128715,0.117218,0.432154,0.200081,1.000000,0.049057,-0.003626,-0.098090,-0.023609
Loan_Amount_Term,-0.100887,-0.127348,-0.087389,-0.090523,-0.032914,-0.069429,0.000951,0.049057,1.000000,0.027392,-0.057004,-0.020291
Credit_History,-0.003570,0.019308,-0.020288,-0.075217,-0.016390,0.028825,0.006564,-0.003626,0.027392,1.000000,-0.008121,0.560936


#### Observation:
<b> Most of feature are poorly or moderately correlated with target variable expect Credit History. </b>
-  Maximum correlation of 0.561 exist between Credit History and Loan status.

 ## 4. Checking Multicollinearity between features using variance_inflation_factor

<b> All features VIF is within permissible limit of 10. 

So No Need to Worry About Multicollinearity.</b>


## 5. Balanceing Imbalanced target feature

In [282]:
df.Loan_Status.value_counts()

Loan_Status
1    398
0    179
Name: count, dtype: int64

<b> As Target variable data is Imbalanced in nature we will need to balance target variable.</b>

### Balancing using SMOTE

In [ ]:
X, Y = split_features_target(df)

<b><em> We have successfully resolved the class imbalanced problem and now all the categories have same data ensuring that the ML model does not get biased towards one category.</em></b>

## Standard Scaling

In [ ]:
X_scale, scaler = scale_features(X)

# Machine Learning Model Building

### Finding best Random state

In [ ]:
best_rs, best_acc = find_best_random_state(
    LogisticRegression, X_scale, Y, test_size=0.3, n_iter=250,
)

## Logistics Regression Model

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scale, Y, random_state=78, test_size=0.3,
)
lr_results = train_and_evaluate(
    LogisticRegression(), X_train, X_test, Y_train, Y_test,
)

In [ ]:
# Decision Tree: default (gini)
dtc_results = train_and_evaluate(
    DecisionTreeClassifier(), X_train, X_test, Y_train, Y_test,
)

# Decision Tree: entropy
dtc_entropy_results = train_and_evaluate(
    DecisionTreeClassifier(criterion='entropy'),
    X_train, X_test, Y_train, Y_test,
)

# Decision Tree: entropy, max_depth=3
dtc_depth3_results = train_and_evaluate(
    DecisionTreeClassifier(criterion='entropy', max_depth=3),
    X_train, X_test, Y_train, Y_test,
)

In [328]:
# Hyperparameter Tuning
# GridSerach 
# RandomizedSerach
# We pass a range of parameter
# Model Evaluate that and Return Best Parameter from them


In [ ]:
param_grid_v1 = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 9],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
gs_v1 = grid_search_evaluate(
    DecisionTreeClassifier(), param_grid_v1,
    X_train, X_test, Y_train, Y_test,
)

In [ ]:
param_grid_v2 = {
    "criterion": ["gini", "entropy"],
    "max_depth": list(range(1, 50, 2)),
    "min_samples_split": list(range(1, 20, 2)),
    "min_samples_leaf": list(range(1, 10)),
}
gs_v2 = grid_search_evaluate(
    DecisionTreeClassifier(), param_grid_v2,
    X_train, X_test, Y_train, Y_test,
)